[요구사항] 
- 학습용 데이터를 이용하여(전처리 필요하면 진행 후) 월간 전력 사용량(electricity)을 예측하는 머신러닝 모델을 학습하시오.
- 학습한 모델을 평가용 데이터에 적용하여 전력 사용량을 예측하고, 예측 결과를 pred 컬럼으로 갖는 데이터프레임을 생성하시오.

In [142]:
import pandas as pd
import numpy as np

In [143]:
# sklearn 데이터분할 함수 임포트
from sklearn.model_selection import train_test_split

# sklearn OneHotEncoder 임포트
from sklearn.preprocessing import OneHotEncoder

# 회귀모델들 임포트 모델비교시 사용
# 3개 거리, 선형, 트리 #
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

# 회귀평가지표 함수 임포트
# 4개 #
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error

모델비교/선택 임포트
- 사람도 공부를 할때 종이,갤럭시탭,아이패드,...아이패드를 사용한다면 화면사이즈, 펜유무,...등 비교 후 선택

평가지표함수 임포트
- 모델 score메서드는 회귀모델은 대표평가지표인 결정계수만 출력(분류모델은 정확도)

타겟이 범주면 분류, 연속형값이면 회귀 모델 선택

In [144]:
train = pd.DataFrame({
    "building_type": [
        "office","office","retail","hospital","school",
        "office","retail","school","hospital","office",
        "retail","office","school","hospital","office",
        "retail","school","office","hospital","retail",
        "office","school","retail","hospital","office",
        "school","retail","office","hospital","office"
    ],

    "area": [
        1200,1800,950,2500,2100,
        1500,1100,2800,3200,1400,
        900,2200,1900,3500,1700,
        1300,2400,2000,2800,1000,
        1600,3000,1250,4000,2300,
        1800,1050,2600,3100,1900
    ],

    "age": [
        5,12,8,20,15,
        7,18,10,25,6,
        14,9,22,30,11,
        4,16,13,19,8,
        10,17,6,28,14,
        9,21,12,18,7
    ],

    "floors": [
        5,8,3,10,4,
        6,3,5,12,7,
        2,9,4,15,6,
        3,5,7,11,4,
        8,6,3,14,9,
        5,2,10,8,6
    ],

    "cooling_area": [
        900,1300,700,1800,1500,
        1100,800,1900,2300,1000,
        650,1600,1300,2500,1200,
        900,1700,1400,2000,700,
        1200,2100,900,2900,1600,
        1300,750,1800,2200,1400
    ],

    "operation_hours": [
        10,12,11,18,8,
        10,13,9,24,11,
        12,14,8,24,10,
        12,9,8,20,11,
        13,7,12,24,10,
        8,13,11,18,12
    ],

    "employees": [
        80,120,60,300,150,
        95,70,180,420,100,
        55,160,130,500,110,
        75,200,140,350,65,
        105,220,80,600,170,
        160,50,190,380,125
    ],

    "electricity": [
        18200,26500,14800,52000,23800,
        22100,17100,30100,68000,20500,
        13200,34200,21400,79000,24900,
        17600,28600,24500,61500,15400,
        23200,32500,16100,91000,35400,
        25100,13900,39200,70500,28900
    ]
})

# 평가 데이터
# 평가 데이터에는 electricity가 없다. -> 예측
test = pd.DataFrame({
    "building_type": [
        "office","retail","hospital","school",
        "office","retail","hospital","office",
        "school","retail"
    ],

    "area": [
        1900,1050,2900,2200,
        1550,1350,3700,2400,
        2600,850
    ],

    "age": [
        9,16,18,12,
        5,11,27,14,
        8,20
    ],

    "floors": [
        7,3,11,5,
        6,4,13,9,
        6,2
    ],

    "cooling_area": [
        1350,750,2100,1600,
        1150,950,2700,1750,
        1900,600
    ],

    "operation_hours": [
        11,12,24,9,
        10,13,24,12,
        8,11
    ],

    "employees": [
        130,65,350,160,
        100,85,550,180,
        210,55
    ]
})

*데이터분할 : "캐글"같은 데이터분석/ML/DL 대회 프랫폼에서 "X_test"변수명이 최종 답안용으로 사용된다. <br>
그래서 보통 2개 학습과 검증(validation)데이터셋의 분할이름을 X_train, X_val, y_train, y_val로 한다. 좀 더 꼼꼼한 모델학습을 위해 3개의 세트로 분할할때는 train, val, test 형태로 사용한다. <br>

In [145]:
train

,building_type,area,age,floors,cooling_area,operation_hours,employees,electricity
0,office,1200,5,5,900,10,80,18200
1,office,1800,12,8,1300,12,120,26500
2,retail,950,8,3,700,11,60,14800
3,hospital,2500,20,10,1800,18,300,52000
4,school,2100,15,4,1500,8,150,23800
5,office,1500,7,6,1100,10,95,22100
6,retail,1100,18,3,800,13,70,17100
7,school,2800,10,5,1900,9,180,30100
8,hospital,3200,25,12,2300,24,420,68000
9,office,1400,6,7,1000,11,100,20500


In [146]:
test

,building_type,area,age,floors,cooling_area,operation_hours,employees
0,office,1900,9,7,1350,11,130
1,retail,1050,16,3,750,12,65
2,hospital,2900,18,11,2100,24,350
3,school,2200,12,5,1600,9,160
4,office,1550,5,6,1150,10,100
5,retail,1350,11,4,950,13,85
6,hospital,3700,27,13,2700,24,550
7,office,2400,14,9,1750,12,180
8,school,2600,8,6,1900,8,210
9,retail,850,20,2,600,11,55


In [147]:
train.info() # 탐색

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   building_type    30 non-null     object
 1   area             30 non-null     int64 
 2   age              30 non-null     int64 
 3   floors           30 non-null     int64 
 4   cooling_area     30 non-null     int64 
 5   operation_hours  30 non-null     int64 
 6   employees        30 non-null     int64 
 7   electricity      30 non-null     int64 
dtypes: int64(7), object(1)
memory usage: 2.0+ KB


In [148]:
train['building_type'].unique()

array(['office', 'retail', 'hospital', 'school'], dtype=object)

In [149]:
test['building_type'].unique()

array(['office', 'retail', 'hospital', 'school'], dtype=object)

In [150]:
train['building_type'].value_counts()

,count
building_type,
office,11
retail,7
hospital,6
school,6


In [151]:
test['building_type'].value_counts()

,count
building_type,
office,3
retail,3
hospital,2
school,2


In [ ]:
from re import X
# 1. X, y 분리 (특성과 타겟 나누기)
# train과 test원본 유지
# X, y -> X_train, y_train, X_val, y_val
# X_test

X = train.drop('electricity', axis=1) # y열을 삭제한 데이터셋 -> DataFrame
y = train['electricity'] # y열만 -> 시리즈

display(X.head())
display(y.head())

X_test = test.copy()

,building_type,area,age,floors,cooling_area,operation_hours,employees
0,office,1200,5,5,900,10,80
1,office,1800,12,8,1300,12,120
2,retail,950,8,3,700,11,60
3,hospital,2500,20,10,1800,18,300
4,school,2100,15,4,1500,8,150


,electricity
0,18200
1,26500
2,14800
3,52000
4,23800


In [ ]:
# 2. 학습 / 검증 데이터 분리 (누수 방지를 위해 인코딩 전 분할)
# 반환값은 언패킹해서 4개의 변수에 대입
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42) # [a,b,c,d]
display(len(X_train))
display(len(X_val))
display(len(y_train))
display(len(y_val))

24

6

24

6

In [ ]:
# 3. 전처리 인코딩
# OneHotEncoder()
# sparse_output=False 반환값을 numpy로
# 새로운 변수추가로 발생하는 에러는 무시
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # sparse_output=False -> fit_transform()반환값이 ndarray

# fit_transform() * 3번(X_train, X_val, X_test)
# 열추가는 데이이터프레임+numpy배열이므로 numpy.stack()나 pandas.concat()사용 * 3번

# 인코딩 학습시 동일한 변수의 학습은(fit) 처음 한번만 -> 뒤에 같은 변수의 인코딩이 동일한 기준으로 인코딩된다
# 1) [학습 데이터] 인코딩 결과 배열 + 나머지 수치형 배열을 그냥 가로로 합체!
# fit - 데이터를 공부하니까 building_type이 4가지 있네
# transform - 그러면 0001, 0010, 0100, 1000 변경할게
X_train_cat = ohe.fit_transform(X_train[['building_type']]) # 인코딩 기준 학습시 fit_transform()
print(X_train_cat)

# 2) [검증 데이터] 동일하게 처리 (fit은 절대 하지 않음!)
# fit - 어 3가지밖에 없네
# transform - 001, 010, 100으로 변경하게
X_val_cat = ohe.transform(X_val[['building_type']]) #  이미 동일한 변수에 대해 학습되어있다면 transform()
print(X_val_cat)

# 3) [테스트 데이터] 동일하게 처리
# fit - 데이터를 공부하니까 building_type이 4가지 있네
# transform - 그러면 0001, 0010, 0100, 1000 변경할게
X_test_cat = ohe.transform(X_test[['building_type']]) #  이미 동일한 변수에 대해 학습되어있다면 transform()
print(X_test_cat)

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 1. 0.]]
[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
[[0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 1. 0.]]


In [155]:
X_train.index

Index([28, 24, 12,  0,  4, 16,  5, 13, 11, 22,  1,  2, 25,  3, 21, 26, 18, 29,
       20,  7, 10, 14, 19,  6],
      dtype='int64')

In [ ]:
# X_train_cat이 ndarray타입이다. DataFrame타입으로 변경 Why? pd.concat([df1, df2])함수를 사용하기 위해서
# concat 같은 인덱스끼리 열결합 -> 생성되는 X_train_cat_df 인덱스를 X_train의 인덱스와 동일
X_train_cat_df = pd.DataFrame(X_train_cat, index=X_train.index, columns=['a','b','c', 'd'])
display(X_train_cat_df)
# building_type 열 drop
X_train_drop = X_train.drop('building_type', axis=1)
display(X_train_drop)

# X_train_drop  +  X_train_cat_df
X_train_encoded = pd.concat([X_train_drop, X_train_cat_df], axis=1)
display(X_train_encoded)

,a,b,c,d
28,1.0,0.0,0.0,0.0
24,0.0,1.0,0.0,0.0
12,0.0,0.0,0.0,1.0
0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0
16,0.0,0.0,0.0,1.0
5,0.0,1.0,0.0,0.0
13,1.0,0.0,0.0,0.0
11,0.0,1.0,0.0,0.0
22,0.0,0.0,1.0,0.0


,area,age,floors,cooling_area,operation_hours,employees
28,3100,18,8,2200,18,380
24,2300,14,9,1600,10,170
12,1900,22,4,1300,8,130
0,1200,5,5,900,10,80
4,2100,15,4,1500,8,150
16,2400,16,5,1700,9,200
5,1500,7,6,1100,10,95
13,3500,30,15,2500,24,500
11,2200,9,9,1600,14,160
22,1250,6,3,900,12,80


,area,age,floors,cooling_area,operation_hours,employees,a,b,c,d
28,3100,18,8,2200,18,380,1.0,0.0,0.0,0.0
24,2300,14,9,1600,10,170,0.0,1.0,0.0,0.0
12,1900,22,4,1300,8,130,0.0,0.0,0.0,1.0
0,1200,5,5,900,10,80,0.0,1.0,0.0,0.0
4,2100,15,4,1500,8,150,0.0,0.0,0.0,1.0
16,2400,16,5,1700,9,200,0.0,0.0,0.0,1.0
5,1500,7,6,1100,10,95,0.0,1.0,0.0,0.0
13,3500,30,15,2500,24,500,1.0,0.0,0.0,0.0
11,2200,9,9,1600,14,160,0.0,1.0,0.0,0.0
22,1250,6,3,900,12,80,0.0,0.0,1.0,0.0


In [157]:
# X_val
X_val_cat_df = pd.DataFrame(X_val_cat, index=X_val.index, columns=['a','b','c', 'd'])
display(X_val_cat_df)
X_val_drop = X_val.drop('building_type', axis=1)
display(X_val_drop)
X_val_encoded = pd.concat([X_val_drop, X_val_cat_df], axis=1)
display(X_val_encoded)

# X_test
X_test_cat_df = pd.DataFrame(X_test_cat, index=X_test.index, columns=['a','b','c', 'd'])
display(X_test_cat_df)
X_test_drop = X_test.drop('building_type', axis=1)
display(X_test_drop)
X_test_encoded = pd.concat([X_test_drop, X_test_cat_df], axis=1)
display(X_test_encoded)

,a,b,c,d
27,0.0,1.0,0.0,0.0
15,0.0,0.0,1.0,0.0
23,1.0,0.0,0.0,0.0
17,0.0,1.0,0.0,0.0
8,1.0,0.0,0.0,0.0
9,0.0,1.0,0.0,0.0


,area,age,floors,cooling_area,operation_hours,employees
27,2600,12,10,1800,11,190
15,1300,4,3,900,12,75
23,4000,28,14,2900,24,600
17,2000,13,7,1400,8,140
8,3200,25,12,2300,24,420
9,1400,6,7,1000,11,100


,area,age,floors,cooling_area,operation_hours,employees,a,b,c,d
27,2600,12,10,1800,11,190,0.0,1.0,0.0,0.0
15,1300,4,3,900,12,75,0.0,0.0,1.0,0.0
23,4000,28,14,2900,24,600,1.0,0.0,0.0,0.0
17,2000,13,7,1400,8,140,0.0,1.0,0.0,0.0
8,3200,25,12,2300,24,420,1.0,0.0,0.0,0.0
9,1400,6,7,1000,11,100,0.0,1.0,0.0,0.0


,a,b,c,d
0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0
2,1.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0
5,0.0,0.0,1.0,0.0
6,1.0,0.0,0.0,0.0
7,0.0,1.0,0.0,0.0
8,0.0,0.0,0.0,1.0
9,0.0,0.0,1.0,0.0


,area,age,floors,cooling_area,operation_hours,employees
0,1900,9,7,1350,11,130
1,1050,16,3,750,12,65
2,2900,18,11,2100,24,350
3,2200,12,5,1600,9,160
4,1550,5,6,1150,10,100
5,1350,11,4,950,13,85
6,3700,27,13,2700,24,550
7,2400,14,9,1750,12,180
8,2600,8,6,1900,8,210
9,850,20,2,600,11,55


,area,age,floors,cooling_area,operation_hours,employees,a,b,c,d
0,1900,9,7,1350,11,130,0.0,1.0,0.0,0.0
1,1050,16,3,750,12,65,0.0,0.0,1.0,0.0
2,2900,18,11,2100,24,350,1.0,0.0,0.0,0.0
3,2200,12,5,1600,9,160,0.0,0.0,0.0,1.0
4,1550,5,6,1150,10,100,0.0,1.0,0.0,0.0
5,1350,11,4,950,13,85,0.0,0.0,1.0,0.0
6,3700,27,13,2700,24,550,1.0,0.0,0.0,0.0
7,2400,14,9,1750,12,180,0.0,1.0,0.0,0.0
8,2600,8,6,1900,8,210,0.0,0.0,0.0,1.0
9,850,20,2,600,11,55,0.0,0.0,1.0,0.0


In [158]:
# 모델생성, 학습, 검증, 평가
model = RandomForestRegressor()
# model = LinearRegression()
# model = KNeighborsRegressor()

# 훈련데이터(X_train_encoded, y_train) 학습
model.fit(X_train_encoded, y_train)
# print(model.coef_) # 선형모델의 속성
# print(model.intercept_) # 선형모델의 속성

RandomForestRegressor()

In [159]:
# 검정, 평가(X_val_encoded, y_val)
val_pred = model.predict(X_val_encoded)
# r2_score(정답지, 예측지)
r2 = r2_score(y_val, val_pred)
MAE = mean_absolute_error(y_val, val_pred)
MSE = mean_squared_error(y_val, val_pred)
RMSE = root_mean_squared_error(y_val, val_pred)

print('r2:', r2)
print('MAE:', MAE)
print('MSE:', MSE)
print('RMSE:', RMSE)

r2: 0.9256712837538268
MAE: 4322.0
MSE: 54951137.333333336
RMSE: 7412.903434777317


In [160]:
# 예측(X_val_encoded, y_test)